<a href="https://colab.research.google.com/github/T-Svitlichna/DTA_Python/blob/main/Proekt_2/notekooks/melbourne_houses_analyse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Melbourne Housing Snapshot

  Dataset was scraped from publicly available results posted every week from Domain.com.au
  This is intended as a static (unchanging) snapshot of https://www.kaggle.com/anthonypino/melbourne-housing-market. It was created in September 2017. Additionally, homes with no Price have been removed.

#Overview
  File informations
- melb_data.csv(2.09 MB)
- number of Rows 13580
- namber of Columns 21


#

| Column name | Description | Example Value |
|---|---|---|
| Suburb | Suburb name | Abbotsford |
| Address | Street address | 85 Turner St |
| Rooms | Number of rooms | 2 |
| Type | h = house/cottage/villa/semi/terrace · u = unit/duplex · t = townhouse · dev site = development site · o res = other residential | h, u, t |
| Price | Price in dollars | 1480000 |
| Method | See note below | S, SP, PI |
| SellerG | Real Estate Agent | Biggin |
| Date | Date sold | 3/12/2016 |
| Distance | Distance from CBD | 2.5 |
| Postcode | Postcode | 3067 |
| Bedroom2 | Scraped # of Bedrooms (from different source) | 2 |
| Bathroom | Number of Bathrooms | 1 |
| Car | Number of carspots | 1 |
| Landsize | Land Size in m² | 202 |
| BuildingArea | Building Size in m² | 126 |
| YearBuilt | Year built | 1970 |
| CouncilArea | Governing council for the area | Yarra City Council |
| Lattitude | Latitude coordinate | -37.7996 |
| Longtitude | Longitude coordinate | 144.9984 |
| Regionname | General Region (West, North West, North, North East etc.) | Northern Metropolitan |
| Propertycount | Number of properties that exist in the suburb | 4019 |

---

**Method values:**
- **S** — property sold
- **SP** — sold prior
- **PI** — passed in
- **PN** — sold prior not disclosed
- **SN** — sold not disclosed
- **NB** — no bid
- **VB** — vendor bid
- **W** — withdrawn prior to auction
- **SA** — sold after auction
- **SS** — sold after auction, price not disclosed
- **N/A** — price or highest bid not available

!pip install numpy pandas matplotlib

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as  plt

FILE_IN = 'https://raw.githubusercontent.com/T-Svitlichna/Melbourne_housing/refs/heads/main/data/melb_data.csv'
FILE_OUT = "data/Melbourne_houses_analyse.csv"

df_origin = pd.read_csv(FILE_IN)



#Primary diagnosis

In [5]:
df_origin.head()
df = df_origin.copy()
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  object 
 1   Address        13580 non-null  object 
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  object 
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  object 
 6   SellerG        13580 non-null  object 
 7   Date           13580 non-null  object 
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  object 
 17  Lattitude      13580 non-null  float64
 18  Longti

,Rooms,Price,Distance,Postcode,Bedroom2,Bathroom,Car,Landsize,BuildingArea,YearBuilt,Lattitude,Longtitude,Propertycount
count,13580.000000,1.358000e+04,13580.000000,13580.000000,13580.000000,13580.000000,13518.000000,13580.000000,7130.000000,8205.000000,13580.000000,13580.000000,13580.000000
mean,2.937997,1.075684e+06,10.137776,3105.301915,2.914728,1.534242,1.610075,558.416127,151.967650,1964.684217,-37.809203,144.995216,7454.417378
std,0.955748,6.393107e+05,5.868725,90.676964,0.965921,0.691712,0.962634,3990.669241,541.014538,37.273762,0.079260,0.103916,4378.581772
min,1.000000,8.500000e+04,0.000000,3000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1196.000000,-38.182550,144.431810,249.000000
25%,2.000000,6.500000e+05,6.100000,3044.000000,2.000000,1.000000,1.000000,177.000000,93.000000,1940.000000,-37.856822,144.929600,4380.000000
50%,3.000000,9.030000e+05,9.200000,3084.000000,3.000000,1.000000,2.000000,440.000000,126.000000,1970.000000,-37.802355,145.000100,6555.000000
75%,3.000000,1.330000e+06,13.000000,3148.000000,3.000000,2.000000,2.000000,651.000000,174.000000,1999.000000,-37.756400,145.058305,10331.000000
max,10.000000,9.000000e+06,48.100000,3977.000000,20.000000,8.000000,10.000000,433014.000000,44515.000000,2018.000000,-37.408530,145.526350,21650.000000


Problem:

Date columns (Transaction Date) as object

Find  dirty  and Null values

In [8]:
print("Missing values:\n")
missing = df.isnull().sum()
missing_pct = (missing/len(df) * 100).round(2)
missing_rep = pd.DataFrame({
    "count": missing,
    "%": missing_pct
})
#print(missing_rep)
missnotnull = missing_rep[missing_rep["%"] > 0]
print(missnotnull)

Missing values:

              count      %
Car              62   0.46
BuildingArea   6450  47.50
YearBuilt      5375  39.58
CouncilArea    1369  10.08


In [ ]:
dirty = ["ERROR", "UNKNOWN", "UNK", "N/A", "?", "-", ""," " "NONE", "NULL"]

for col in ['Suburb', 'Address', 'Rooms', 'Type', 'Price', 'Method', 'SellerG',
       'Date', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car',
       'Landsize', 'BuildingArea', 'YearBuilt', 'CouncilArea', 'Lattitude',
       'Longtitude', 'Regionname', 'Propertycount']:
    count = df[col].astype(str).str.upper().isin(dirty).sum()
    nan = df[col].isna().sum()
    display(f"{col}: брудних={count}, NaN={nan}")

'Suburb: брудних=0, NaN=0'

'Address: брудних=0, NaN=0'

'Rooms: брудних=0, NaN=0'

'Type: брудних=0, NaN=0'

'Price: брудних=0, NaN=0'

...

'CouncilArea: брудних=0, NaN=1369'

'Lattitude: брудних=0, NaN=0'

'Longtitude: брудних=0, NaN=0'

'Regionname: брудних=0, NaN=0'

'Propertycount: брудних=0, NaN=0'

Dirty value didn't find

# BUILDING AREA: MISSING VALUE ANALYSIS & IMPUTATION STRATEGY

In [9]:
(df["BuildingArea"] == 0).sum()

np.int64(17)

STEP 1: Replace 0 with NaN
# 0 is not a valid building area — treat as missing

In [11]:
df['BuildingArea'] = df['BuildingArea'].replace(0, np.nan)

print(f"Missing values: {df['BuildingArea'].isna().sum()} ({df['BuildingArea'].isna().mean()*100:.1f}%)")

Missing values: 6467 (47.6%)


# STEP 2: Split into complete and missing subsets
# We analyse only complete data to avoid biased conclusions

In [12]:
df_complete = df[df['BuildingArea'].notna()].copy()
df_missing  = df[df['BuildingArea'].isna()].copy()

print(f"Complete rows : {len(df_complete)}")
print(f"Missing rows  : {len(df_missing)}")

Complete rows : 7113
Missing rows  : 6467


In [ ]:
df_complete.BuildingArea.describe()

,BuildingArea
count,7113.000000
mean,152.330851
std,541.609671
min,1.000000
25%,93.000000
50%,126.000000
75%,174.000000
max,44515.000000


STEP 3: Correlation with numeric features
# Higher value = stronger relationship with BuildingArea

https://raw.githubusercontent.com/T-Svitlichna/Melbourne_housing/refs/heads/main/data/melb_data.csv